# Hash Maps & Hash Tables in Python — Implementation, Collisions & Algorithmic Patterns

> **Topic:** Hash Maps & Hash Tables | **Folder:** Data Structures & Algorithms

A **Hash Map (Hash Table)** is a data structure that implements an associative array abstract data type,
mapping **keys to values**.

Using a **hash function** to compute index buckets, Hash Maps achieve **$O(1)$ average time complexity**
for insertion, lookup, and deletion.

---

## Table of Contents
1. [Hash Table Concepts & Complexity](#1.-Hash-Table-Concepts-&-Complexity)
2. [Hash Functions & The Built-in `hash()` Function](#2.-Hash-Functions-&-The-Built-in-`hash()`-Function)
3. [Collision Resolution Strategies (Separate Chaining vs. Open Addressing)](#3.-Collision-Resolution-Strategies-(Separate-Chaining-vs.-Open-Addressing))
4. [Building a Hash Map from Scratch in Python](#4.-Building-a-Hash-Map-from-Scratch-in-Python)
5. [Python `dict` & `set` Under the Hood (Compact Hash Tables)](#5.-Python-`dict`-&-`set`-Under-the-Hood-(Compact-Hash-Tables))
6. [Algorithmic Pattern 1: Frequency Counter / Histogram](#6.-Algorithmic-Pattern-1:-Frequency-Counter-/-Histogram)
7. [Algorithmic Pattern 2: Two Sum (Complement Lookup)](#7.-Algorithmic-Pattern-2:-Two-Sum-(Complement-Lookup))
8. [Algorithmic Pattern 3: Group Anagrams](#8.-Algorithmic-Pattern-3:-Group-Anagrams)
9. [Algorithmic Pattern 4: Subarray Sum Equals K (Prefix Sum + Hash Map)](#9.-Algorithmic-Pattern-4:-Subarray-Sum-Equals-K-(Prefix-Sum-+-Hash-Map))
10. [Algorithmic Pattern 5: LRU Cache (Hash Map + Doubly Linked List)](#10.-Algorithmic-Pattern-5:-LRU-Cache-(Hash-Map-+-Doubly-Linked-List))
11. [Quick Reference Card](#11.-Quick-Reference-Card)


---
## 1. Hash Table Concepts & Complexity

| Operation | Average Case | Worst Case (All Collisions) | Notes |
|-----------|--------------|-----------------------------|-------|
| **Lookup `get(k)`** | $O(1)$ | $O(n)$ | Direct index computation |
| **Insertion `put(k, v)`** | $O(1)$ | $O(n)$ | Rehash / Resizing is amortized $O(1)$ |
| **Deletion `delete(k)`** | $O(1)$ | $O(n)$ | Bucket removal |
| **Space Complexity** | $O(n)$ | $O(n)$ | Stores key-value pairs |


In [ ]:
# Basic dict usage demonstrating O(1) operations
user_scores = {"Alice": 95, "Bob": 88, "Charlie": 72}

# O(1) lookup
print("Alice's score:", user_scores.get("Alice"))

# O(1) insertion
user_scores["Dave"] = 91

# O(1) membership check
print("Is Eve in dictionary?", "Eve" in user_scores)


---
## 2. Hash Functions & The Built-in `hash()` Function

A **hash function** maps arbitrary data (strings, integers, tuples) to a fixed-size integer value.

### Requirements of a Hash Function:
1. **Deterministic**: `hash(x)` must always yield the exact same integer for identical inputs.
2. **Fast**: Must compute in $O(1)$ or $O(L)$ where $L$ is input length.
3. **Uniform Distribution**: Spreads keys evenly across available buckets to minimize collisions.

> **Note**: Only **immutable (hashable)** types (`int`, `str`, `float`, `tuple`, `frozenset`) can be used as keys!


In [ ]:
# Exploring Python's built-in hash() function
print("hash(42)          :", hash(42))
print("hash('hello')     :", hash("hello"))
print("hash((1, 2, 3))   :", hash((1, 2, 3)))

# Mutable types (lists, dicts, sets) are NOT hashable!
try:
    hash([1, 2, 3])
except TypeError as e:
    print(f"\nTypeError caught: {e}")


---
## 3. Collision Resolution Strategies

A **collision** occurs when two distinct keys hash to the same bucket index ($h(k_1) = h(k_2)$).

### 1. Separate Chaining (Open Hashing)
Each bucket contains a linked list or dynamic array storing all key-value pairs that hash to that index.

### 2. Open Addressing (Closed Hashing)
All elements reside directly within the hash table array itself.
- **Linear Probing**: Searches consecutive slots $h(k) + 1, h(k) + 2, \dots$
- **Quadratic Probing**: Searches $h(k) + 1^2, h(k) + 2^2, \dots$
- **Double Hashing**: Uses a second hash function $h_2(k)$ for step size.


---
## 4. Building a Hash Map from Scratch in Python

Below is a complete `HashMap` implementation using **Separate Chaining** with automatic resizing
when the **load factor** ($\alpha = \text{size} / \text{capacity}$) exceeds 0.75.


In [ ]:
class HashMap:
    def __init__(self, initial_capacity=8, load_factor=0.75):
        self.capacity = initial_capacity
        self.load_factor = load_factor
        self.size = 0
        self.buckets = [[] for _ in range(self.capacity)]

    def _hash(self, key):
        return hash(key) % self.capacity

    def put(self, key, value):
        index = self._hash(key)
        bucket = self.buckets[index]
        
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)  # Update existing key
                return
                
        bucket.append((key, value))      # Insert new pair
        self.size += 1
        
        # Check load factor threshold for auto-resizing
        if self.size / self.capacity >= self.load_factor:
            self._resize(self.capacity * 2)

    def get(self, key, default=None):
        index = self._hash(key)
        bucket = self.buckets[index]
        for k, v in bucket:
            if k == key: return v
        return default

    def remove(self, key):
        index = self._hash(key)
        bucket = self.buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                del bucket[i]
                self.size -= 1
                return True
        return False

    def _resize(self, new_capacity):
        old_buckets = self.buckets
        self.capacity = new_capacity
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:
            for k, v in bucket:
                self.put(k, v)

    def __getitem__(self, key): return self.get(key)
    def __setitem__(self, key, value): self.put(key, value)
    def __delitem__(self, key): self.remove(key)
    def __contains__(self, key): return self.get(key) is not None
    def __len__(self): return self.size

# Testing Custom HashMap
hm = HashMap(initial_capacity=4)
hm["name"] = "Alice"
hm["role"] = "Engineer"
hm["city"] = "Paris"
hm["country"] = "France"  # Triggers auto-resize

print(f"HashMap length: {len(hm)}, capacity: {hm.capacity}")
print("hm['name']    :", hm["name"])
print("hm['city']    :", hm["city"])
print("'role' in hm  :", "role" in hm)
del hm["role"]
print("'role' in hm  :", "role" in hm)


---
## 5. Python `dict` & `set` Under the Hood

Since Python 3.6, CPython uses a **compact hash table layout**:
1. A dense array `entries = [key, hash, value]` that maintains **insertion order**.
2. A sparse hash table index array pointing to indices in `entries`.

This architectural overhaul reduced dictionary memory consumption by 20%–25%!


In [ ]:
# CPython preserves insertion order
d = {}
d["z"] = 1
d["a"] = 2
d["m"] = 3

print("Dict keys in insertion order:", list(d.keys()))


---
## 6. Algorithmic Pattern 1: Frequency Counter / Histogram

Counts frequency of elements in $O(n)$ time using Hash Map or `collections.Counter`.


In [ ]:
from collections import Counter

text = "abracadabra"

# Manual Hash Map Frequency Counter
freq = {}
for char in text:
    freq[char] = freq.get(char, 0) + 1
print("Manual Freq Counter:", freq)

# Built-in Counter
counts = Counter(text)
print("Counter top 3      :", counts.most_common(3))


---
## 7. Algorithmic Pattern 2: Two Sum (Complement Lookup)

Finds indices of two numbers that add up to `target` in **$O(n)$ time** instead of $O(n^2)$ nested loops.


In [ ]:
def two_sum(nums, target):
    seen = {}  # val -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

numbers = [2, 7, 11, 15]
target_val = 9
print(f"Two Sum indices for target {target_val}:", two_sum(numbers, target_val))


---
## 8. Algorithmic Pattern 3: Group Anagrams

Groups words that contain the exact same character frequencies using sorted tuples or character frequency tuples as Hash Map keys.


In [ ]:
from collections import defaultdict

def group_anagrams(words):
    groups = defaultdict(list)
    for word in words:
        # Signature key: sorted string characters tuple
        signature = tuple(sorted(word))
        groups[signature].append(word)
    return list(groups.values())

word_list = ["eat", "tea", "tan", "ate", "nat", "bat"]
print("Grouped Anagrams:", group_anagrams(word_list))


---
## 9. Algorithmic Pattern 4: Subarray Sum Equals K (Prefix Sum + Hash Map)

Finds the number of contiguous subarrays that sum to $k$ in **$O(n)$ time** using a Prefix Sum Frequency Hash Map.


In [ ]:
def subarray_sum_equals_k(nums, k):
    count = 0
    curr_sum = 0
    prefix_counts = {0: 1}  # Base case: empty subarray sum = 0
    
    for x in nums:
        curr_sum += x
        # If (curr_sum - k) exists in prefix_counts, we found matching subarrays!
        if (curr_sum - k) in prefix_counts:
            count += prefix_counts[curr_sum - k]
        prefix_counts[curr_sum] = prefix_counts.get(curr_sum, 0) + 1
        
    return count

arr = [1, 1, 1]
print("Subarrays summing to 2 in [1, 1, 1]:", subarray_sum_equals_k(arr, 2))


---
## 10. Algorithmic Pattern 5: LRU Cache (Hash Map + Doubly Linked List)

Implements a **Least Recently Used (LRU) Cache** supporting `get(key)` and `put(key, value)` in **$O(1)$ time**
by combining a Hash Map (for $O(1)$ lookup) with a Doubly Linked List (for $O(1)$ node relocation).


In [ ]:
class LRUNode:
    def __init__(self, key=0, val=0):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key -> LRUNode
        # Dummy head and tail nodes
        self.head = LRUNode()
        self.tail = LRUNode()
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node):
        prev, nxt = node.prev, node.next
        prev.next = nxt
        nxt.prev = prev

    def _add_to_head(self, node):
        nxt = self.head.next
        node.prev = self.head
        node.next = nxt
        self.head.next = node
        nxt.prev = node

    def get(self, key: int) -> int:
        if key in self.cache:
            node = self.cache[key]
            self._remove(node)
            self._add_to_head(node)  # Move to head (recently used)
            return node.val
        return -1

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self._remove(self.cache[key])
        node = LRUNode(key, value)
        self.cache[key] = node
        self._add_to_head(node)
        if len(self.cache) > self.capacity:
            lru = self.tail.prev  # Evict least recently used
            self._remove(lru)
            del self.cache[lru.key]

# Testing LRUCache
lru = LRUCache(capacity=2)
lru.put(1, 10)
lru.put(2, 20)
print("Get key 1:", lru.get(1))    # Returns 10
lru.put(3, 30)                    # Evicts key 2
print("Get key 2:", lru.get(2))    # Returns -1 (evicted!)


---
## 11. Quick Reference Card


In [ ]:
# ==================================================================
# HASH MAPS – QUICK REFERENCE
# ==================================================================
from collections import Counter, defaultdict, OrderedDict

# --- Frequency Counter ---
counts = Counter([1, 2, 2, 3, 3, 3])
print("Most common:", counts.most_common(1))

# --- Grouping with defaultdict ---
groups = defaultdict(list)
groups["even"].append(2)

# --- LRU Cache via OrderedDict ---
od = OrderedDict()
od["a"] = 1; od["b"] = 2
od.move_to_end("a")  # Move 'a' to most recently used end
print("OrderedDict:", list(od.keys()))


---
## Summary

| Technique / Structure | Time Complexity | Applications |
|-----------------------|-----------------|--------------|
| **Hash Map Lookup** | $O(1)$ avg | Instant key-value retrieval |
| **Frequency Counter** | $O(n)$ | Histograms, anagram checks |
| **Complement Lookup** | $O(n)$ | Two Sum, pair checking |
| **Prefix Sum + Hash Map** | $O(n)$ | Subarray sum equality algorithms |
| **LRU Cache** | $O(1)$ `get`/`put` | Memory caching, buffer pools |

---
*Next up: **Stacks & Queues***
